# Full metric-suite smoke review

Review the three development-only Berzelius evaluations.

The notebook validates outputs before it shows metrics or images.

In [ ]:
# Locate the newest complete three-task smoke directory.
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone

import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd().resolve()
if repo_root.name != "mem_param_clean":
    repo_root = next(
        parent for parent in repo_root.parents if parent.name == "mem_param_clean"
    )
sys.path.insert(0, str(repo_root))

from mem_param_src.runners.artifacts import read_json
from mem_param_src.runners.full_suite_validate import validate_full_suite

smoke_parent = repo_root / "outputs" / "smoke" / "full_metric_suite_512"
# Select only batches that contain all three result directories.
candidates = [
    path
    for path in smoke_parent.iterdir()
    if all((path / name).is_dir() for name in ("unet32", "mnist32", "dit32"))
]
batch_root = sorted(candidates)[-1]
result_roots = {
    name: batch_root / name for name in ("unet32", "mnist32", "dit32")
}
batch_root

In [ ]:
# Validate checkpoint counts, metrics, sample shapes, reuse, and plots.
expected_counts = {"unet32": 72, "mnist32": 20, "dit32": 72}
acceptance = {
    name: validate_full_suite(
        root,
        expected_checkpoints=expected_counts[name],
        sample_count=512,
    )
    for name, root in result_roots.items()
}
display(pd.DataFrame.from_dict(acceptance, orient="index"))

In [ ]:
# Load one combined metric table for each evaluated model.
metric_tables = {
    name: pd.read_csv(root / "combined" / "checkpoint_metrics.csv")
    for name, root in result_roots.items()
}
for name, table in metric_tables.items():
    print(name)
    display(table.tail(20))

In [ ]:
# Show quality, memorization, loss, range, and runtime trajectories.
trajectory_names = (
    "quality_trajectories.png",
    "heldout_loss_trajectories.png",
    "raw_range_diagnostics.png",
    "runtime_diagnostics.png",
)
for name, root in result_roots.items():
    print(name)
    paths = [root / "plots" / item for item in trajectory_names]
    paths.extend(sorted((root / "plots").glob("memorization_*.png")))
    for path in paths:
        display(Image(filename=str(path)))

In [ ]:
# Show sample grids at the four fixed optimizer steps.
for name, root in result_roots.items():
    print(name)
    for path in sorted((root / "plots").glob("samples_step_*.png")):
        display(Image(filename=str(path)))

In [ ]:
# Show final-checkpoint nearest-neighbor panels for each feature space.
for name, root in result_roots.items():
    print(name)
    for path in sorted((root / "plots").glob("nearest_neighbors_*.png")):
        display(Image(filename=str(path)))

In [ ]:
# List the main immutable inputs and development result paths.
artifact_rows = []
for name, root in result_roots.items():
    manifest = read_json(root / "eval_manifest.json")
    artifact_rows.append(
        {
            "task": name,
            "result": str(root),
            "source_run": manifest["source_run"],
            "noise": manifest["noise"]["path"],
            "reference": manifest.get("reference_artifact", {}).get("path"),
            "accepted": str(root / "ACCEPTED.json"),
        }
    )
display(pd.DataFrame(artifact_rows))

## Optional checkpoint rerun

Keep the next cell disabled during normal review.

Enable it only when you need an independent checkpoint replay.

In [ ]:
# Optionally rerun one checkpoint with the canonical 512-row noise prefix.
RUN_OPTIONAL = False
selected_task = "unet32"
selected_step = 1_000_000

if RUN_OPTIONAL:
    source_manifest = read_json(result_roots[selected_task] / "eval_manifest.json")
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    # Keep notebook work separate from the accepted Berzelius result.
    rerun_root = batch_root / "notebook_reruns" / timestamp / selected_task
    command = [
        sys.executable,
        "-m",
        "mem_param_src.runners.image_eval",
        "--run-dir",
        source_manifest["source_run"],
        "--checkpoints",
        str(selected_step),
        "--metrics",
        "full",
        "--sample-count",
        "512",
        "--loss-count",
        "512",
        "--noise-artifact",
        source_manifest["noise"]["path"],
        "--assets-manifest",
        str(repo_root / "outputs" / "model_assets" / "feature_f_mem" / "manifest.json"),
        "--output-dir",
        str(rerun_root),
        "--write-plots",
        "--device",
        "cuda",
    ]
    reference = source_manifest.get("reference_artifact")
    if reference is not None:
        command.extend(["--reference-artifact", reference["path"]])
    subprocess.run(command, cwd=repo_root, check=True)
    print(rerun_root)